# Notebook 06 — Trace storage and retrieval (Gate 4A)

Locate where the context stores the state and how the query retrieves it. Fast path checks the retrieval estimator + attention-row normalization invariant.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="06", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220637_smoke_eagle_06_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
attention rows sum to 1 (eager) and retrieval score responds to target-aligned writes.

In [4]:
from subliminal_icl import routing as rt
rng = np.random.default_rng(3); H, T, d = 4, 6, 16
attn = rng.random((H, T, T)); attn /= attn.sum(axis=-1, keepdims=True)
norm_ok = rt.check_attention_rows_normalized(attn)
v = rng.standard_normal(d); v /= np.linalg.norm(v)
writes = rng.standard_normal((H, T, d))
# make source position 2 write strongly along v
writes[:, 2, :] += 3.0 * v
score_with = rt.attention_weighted_retrieval(attn, writes, v, query_index=T-1, source_indices=[0,1,2,3])
writes0 = rng.standard_normal((H, T, d))
score_without = rt.attention_weighted_retrieval(attn, writes0, v, query_index=T-1, source_indices=[0,1,2,3])
print("attn rows normalized:", norm_ok, "retrieval with:", round(score_with,3), "without:", round(score_without,3))

attn rows normalized: True retrieval with: 2.235 without: -0.025


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("attn_rows_normalized", norm_ok, "rows sum to 1"),
          ("retrieval_responds", abs(score_with) > abs(score_without), "target-aligned writes raise retrieval score")]
gs = run_gate_checks("gate_04a_storage_retrieval", "Storage + retrieval pathway exists", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,attn_rows_normalized,PASS,rows sum to 1
1,retrieval_responds,PASS,target-aligned writes raise retrieval score


GATE gate_04a_storage_retrieval -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.